# Encrypted RAG with enVector, GCS, and Document AI

This tutorial demonstrates a fully GCP-native Encrypted Retrieval-Augmented Generation (RAG) pipeline.

## Trust Boundaries

The design has three distinct zones:

```
+-----------------------------+     +-----------------------------+
|  Customer Trusted Zone      |     |  Vertex AI Trusted Zone     |
|                             |     |                             |
|  GCS Bucket (raw documents) |     |  text-embedding-004         |
|  pyenvector (HE key)        |     |  Document AI Layout Parser  |
|  pyenvector (decrypt)       |     |  Gemini 2.0 Flash           |
|                             |     |                             |
+-----------------------------+     +-----------------------------+
         |                                      |
         |  plaintext stays inside              |  plaintext stays inside
         |  trusted zones only                  |  trusted zones only
         |                                      |
         +----------> ciphertext only <----------+
                             |
               +-------------v-------------+
               |  enVector Server          |
               |  (UNTRUSTED)              |
               |                           |
               |  stores ciphertext        |
               |  searches on ciphertext   |
               |  never decrypts data      |
               +---------------------------+
```

**GCS and Vertex AI** are services the customer calls directly from their own GCP project — plaintext documents and embeddings never leave the customer's control.
**enVector** is the operator-managed compute plane. It stores the document corpus as ciphertext and runs similarity search over that encrypted index, returning encrypted scores it cannot decrypt. Even if the enVector server is fully compromised, the stored corpus remains encrypted — without the HE private key (which never leaves the customer's trusted zone), an attacker cannot recover the original vectors.

The boundary is enforced by `pyenvector`, which encrypts the document vectors client-side before they cross into the untrusted zone.

## Why FHE Makes the Difference

Every vector search engine — including enVector — keeps index vectors resident in memory to serve low-latency queries. In conventional solutions this means plaintext vectors are loaded into the server's address space: a memory dump, a rogue process, or a compromised host exposes the entire corpus.

enVector stores and searches **fully homomorphic encrypted (FHE) ciphertext** in memory. Similarity scores are computed directly on ciphertext without ever decrypting — a property unique to FHE. Even with unrestricted access to the server's memory or storage, an attacker sees only ciphertext and cannot recover the underlying vectors. Decryption is only possible with the HE private key, which never leaves the customer's trusted zone.

## Prerequisites

Before running this notebook, ensure the following are in place:

- **enVector** deployed from GCP Marketplace. Set the `ENVECTOR_ADDRESS` environment variable to the service endpoint (e.g., `my-envector-host:50050`).
- **GCP project** with the following APIs enabled:
  - Vertex AI API
  - Document AI API
  - Cloud Storage API
- **Authentication**: run `gcloud auth application-default login` so the notebook can call GCP services with your credentials.
- **Environment variables** (set before starting the kernel):
  - `ENVECTOR_ADDRESS` — enVector service endpoint
  - `GOOGLE_CLOUD_PROJECT` — your GCP project ID
  - `GOOGLE_CLOUD_LOCATION` — GCP region (e.g., `us-central1`)

> **Note on Document AI processor location**: Document AI processors are only available in the multi-region locations `us` and `eu` (not regional locations such as `us-central1`). If `GOOGLE_CLOUD_LOCATION` is a region Document AI does not serve, set the separate `DOCAI_LOCATION` environment variable to `us` (or `eu`).

## Install Dependencies

In [ ]:
%pip install pyenvector google-cloud-aiplatform google-cloud-storage google-cloud-documentai google-genai

## Configuration

All parameters are read from environment variables so that no credentials or project IDs need to be hard-coded in the notebook.

In [ ]:
import os
import time

ENVECTOR_ADDRESS  = os.getenv("ENVECTOR_ADDRESS", "localhost:50050")
GCP_PROJECT       = os.getenv("GOOGLE_CLOUD_PROJECT", "your-project-id")
GCP_LOCATION      = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")
# Document AI is only available in the multi-region locations "us" and "eu"
# (not regional locations such as us-central1). Override DOCAI_LOCATION if needed.
DOCAI_LOCATION    = os.getenv("DOCAI_LOCATION", "us")
KEY_PATH          = "./keys"
KEY_ID            = "gcp_docai_rag_key"
INDEX_NAME        = "gcp_docai_rag_index"
BUCKET_NAME       = f"{GCP_PROJECT}-envector-demo-{int(time.time())}"

print(f"GCP project  : {GCP_PROJECT}")
print(f"GCP location : {GCP_LOCATION}")
print(f"DocAI location: {DOCAI_LOCATION}")
print(f"GCS bucket   : {BUCKET_NAME}")
print(f"enVector     : {ENVECTOR_ADDRESS}")

In [ ]:
# Prerequisite check: required GCP APIs enabled + Application Default Credentials present.
import subprocess
import google.auth

REQUIRED_APIS = [
    "aiplatform.googleapis.com",   # Vertex AI
    "documentai.googleapis.com",   # Document AI
    "storage.googleapis.com",      # Cloud Storage
]
_enabled = subprocess.run(
    ["gcloud", "services", "list", "--enabled",
     f"--project={GCP_PROJECT}", "--format=value(config.name)"],
    capture_output=True, text=True,
).stdout.split()
for _api in REQUIRED_APIS:
    if _api in _enabled:
        print(f"ENABLED   {_api}")
    else:
        print(f"DISABLED  {_api}  -> gcloud services enable {_api} --project={GCP_PROJECT}")

try:
    google.auth.default()
    print("ADC OK")
except Exception:
    print("ADC missing -> run: gcloud auth application-default login")

## Imports and Initialization

Initialize the Vertex AI client and the enVector SDK. Both are required before any embedding or index operation.

In [ ]:
import numpy as np
from typing import List, Union

import vertexai
from vertexai.language_models import TextEmbeddingModel
from vertexai.generative_models import GenerativeModel
from google.cloud import storage
from google.cloud import documentai
import pyenvector as ev

vertexai.init(project=GCP_PROJECT, location=GCP_LOCATION)

ev.init(
    address=ENVECTOR_ADDRESS,
    key_path=KEY_PATH,
    key_id=KEY_ID,
)

print("Vertex AI and enVector initialized.")


# Gemini model availability varies by project / region / time, so discover it
# at runtime instead of hard-coding. Use $GEMINI_MODEL if set, otherwise the
# first available text-capable flash model.
from google import genai

_genai_client = genai.Client(vertexai=True, project=GCP_PROJECT, location=GCP_LOCATION)
_available_gemini = [
    m.name.split("/")[-1]
    for m in _genai_client.models.list()
    if "gemini" in (m.name or "").lower() and "flash" in m.name.lower()
    and not any(x in m.name.lower() for x in
                ("tts", "image", "preview", "computer-use", "embedding", "live"))
]
print("Available Gemini models:", _available_gemini)
GEMINI_MODEL = os.getenv("GEMINI_MODEL") or (
    _available_gemini[0] if _available_gemini else "gemini-2.5-flash"
)
print("Using Gemini model:", GEMINI_MODEL)

## Upload Documents to GCS

We create a GCS bucket and upload three sample enterprise documents as plain-text files. These represent the kind of internal content an organization would want to keep encrypted: a product changelog, a security policy excerpt, and an on-call runbook snippet.

In a real deployment you would point this step at your existing GCS bucket and document set.

In [ ]:
SAMPLE_DOCUMENTS = {
    "doc1-changelog.txt": """\
Product Changelog - AcmeSaaS Platform

v2.4.0 (2024-11-15)
Added support for custom HMAC signing keys for webhook payloads. Operators can now
rotate signing keys independently per tenant without affecting other integrations.
A 24-hour grace period allows old keys to remain valid during the transition window.

v2.3.1 (2024-10-02)
Fixed a race condition in the token refresh flow that caused intermittent HTTP 401
errors for long-running sessions exceeding 6 hours. The root cause was a missing
mutex around the in-memory token cache. Sessions are now refreshed atomically and
the fix is backported to the v2.3 release branch.

v2.3.0 (2024-09-10)
Introduced per-tenant rate limit configuration via the Management API. Previously,
rate limits could only be changed by the support team through internal tooling.
Tenant admins can now self-serve limit increases up to the plan maximum.

v2.2.0 (2024-07-22)
Added BigQuery export for audit logs. Configurable retention windows (30, 90, 180,
or 365 days) are available on Business and Enterprise plans. Logs include user
identity, action type, resource ID, source IP, and timestamp with microsecond
precision. Exports are incremental and use a partitioned table schema.
""",
    "doc2-security-policy.txt": """\
Internal Security Policy - AcmeSaaS Platform (Excerpt)

Data Encryption
All customer data at rest is encrypted with AES-256-GCM. Encryption keys are
stored in Google Cloud KMS and are rotated automatically every 90 days. Key
rotation does not require downtime; the platform re-encrypts data in the background
using the new key while the old key remains valid for decryption during the
transition period.

Access Logging and Retention
All access to production systems is logged. Logs are retained for a minimum of
one year and are immutable once written. Access log integrity is verified daily
via hash chaining.

Compliance and Audits
A SOC 2 Type II audit is conducted annually by an independent third-party firm.
Penetration testing is performed every six months. The most recent test results
and remediation timelines are documented in the internal security tracker.

Employee Access Controls
Access to production systems requires multi-factor authentication (MFA) and is
granted on a least-privilege basis. Access rights are reviewed quarterly. Any
access that has not been exercised in 60 days is automatically revoked pending
re-approval by the employee's manager.

Data Residency
Customer data is processed and stored exclusively in the region selected at
account creation. Cross-region transfer requires explicit written consent from
the customer and a documented data processing agreement amendment.
""",
    "doc3-runbook.txt": """\
On-Call Runbook - enVector Search Service

High Search Latency (p99 > 2s)
1. Check enVector compute pod CPU utilization in the GKE workload dashboard.
2. If CPU is above 80%, scale the compute deployment horizontally:
   kubectl scale deployment envector-compute --replicas=<current+2>
3. If CPU is normal, check the encryption worker queue depth. A queue depth above
   500 indicates a bottleneck in the FHE evaluation step.
4. Verify that the HPA (Horizontal Pod Autoscaler) is not stuck in a cooldown
   window. If so, trigger a manual scale event.

Low Insert Throughput (< 500 vectors/s)
1. Check the MAX_ENCRYPT_WORKERS setting in the compute pod ConfigMap.
2. If workers < 8, increase to 8 and roll the deployment.
3. If workers are already at the maximum, check for GCS read bottlenecks if
   inserts are sourced from a GCS pipeline.

Missing Search Results
1. Verify the shard replication factor is set to at least 2 in the index config.
2. Run the consistency check tool:
   envector-admin consistency-check --index <index_name>
3. If shards are missing, trigger a rebalance.

Escalation Path
L1 (on-call engineer) -> L2 (backend team Slack: #backend-oncall) ->
L3 (infra team Slack: #infra-oncall). All incidents must be logged in the
incident tracker within 15 minutes of detection. Use the INCIDENT template.
""",
}

In [ ]:
gcs_client = storage.Client(project=GCP_PROJECT)

# Create bucket in the configured region
bucket = gcs_client.create_bucket(BUCKET_NAME, location=GCP_LOCATION)
print(f"Created GCS bucket: gs://{BUCKET_NAME}")

# Upload each sample document
gcs_uris = []
for filename, content in SAMPLE_DOCUMENTS.items():
    blob = bucket.blob(filename)
    blob.upload_from_string(content, content_type="text/plain")
    uri = f"gs://{BUCKET_NAME}/{filename}"
    gcs_uris.append(uri)
    print(f"Uploaded: {uri}")

print(f"\nTotal documents uploaded: {len(gcs_uris)}")

## Parse Documents with Document AI

Document AI Layout Parser processes each document and returns semantically coherent chunks based on the document's logical structure (sections, paragraphs, tables). This removes the need for hand-tuned character or token splitters.

The cell below creates a Layout Parser processor in your project if one does not already exist, then processes each GCS file. For each document, we extract chunks using the `text_anchor` spans that Document AI provides, which reference character offsets into the full document text.

In [ ]:
docai_client = documentai.DocumentProcessorServiceClient(
    client_options={"api_endpoint": f"{DOCAI_LOCATION}-documentai.googleapis.com"}
)

parent = docai_client.common_location_path(GCP_PROJECT, DOCAI_LOCATION)
processor_display_name = "envector-layout-parser"

# Check whether a Layout Parser processor already exists to avoid duplicates
existing_processor = None
for proc in docai_client.list_processors(parent=parent):
    if (
        proc.type_ == "LAYOUT_PARSER_PROCESSOR"
        and proc.display_name == processor_display_name
    ):
        existing_processor = proc
        break

if existing_processor is not None:
    processor = existing_processor
    print(f"Using existing processor: {processor.name}")
else:
    processor = docai_client.create_processor(
        parent=parent,
        processor=documentai.Processor(
            type_="LAYOUT_PARSER_PROCESSOR",
            display_name=processor_display_name,
        ),
    )
    print(f"Created processor: {processor.name}")

In [ ]:
MIN_CHUNK_CHARS = 50  # discard very short fragments


def extract_chunks_from_document(document: documentai.Document) -> List[str]:
    """Extract non-trivial text chunks from a Document AI response.

    Layout Parser populates document.chunked_document.chunks, each of which
    carries a text_anchor with character-offset spans into document.text.
    We use those spans to reconstruct the chunk text faithfully.
    """
    full_text = document.text
    chunks = []

    if document.chunked_document and document.chunked_document.chunks:
        for chunk in document.chunked_document.chunks:
            # Reconstruct chunk text from text_anchor spans
            chunk_text_parts = []
            for segment in chunk.source_block_ids:
                # source_block_ids reference blocks; use text_anchor when available
                pass  # handled below via chunk.content

            # chunk.content holds the concatenated text for the chunk
            text = getattr(chunk, "content", "").strip()

            # Fallback: reconstruct from text_anchor spans on the chunk's page_span
            if not text and chunk.page_span:
                start = chunk.page_span.page_start
                end = chunk.page_span.page_end
                # page_span is in page numbers; use full_text as the best fallback
                text = full_text.strip()

            if len(text) >= MIN_CHUNK_CHARS:
                chunks.append(text)
    else:
        # Fallback: split by double newlines (paragraph-level)
        for para in full_text.split("\n\n"):
            para = para.strip()
            if len(para) >= MIN_CHUNK_CHARS:
                chunks.append(para)

    return chunks


def process_gcs_document(gcs_uri: str, mime_type: str = "text/plain") -> List[str]:
    """Run Document AI Layout Parser on a GCS object and return text chunks.

    Layout Parser returns a deterministic 500 INTERNAL for text/plain input,
    so plain text is wrapped in minimal HTML and sent as text/html.
    """
    bucket_name, _, blob_name = gcs_uri[len("gs://"):].partition("/")
    raw = storage.Client(project=GCP_PROJECT).bucket(bucket_name).blob(blob_name).download_as_bytes()

    if mime_type == "text/plain":
        import html as _html
        body = "".join(
            f"<p>{_html.escape(line)}</p>"
            for line in raw.decode("utf-8", "replace").splitlines()
            if line.strip()
        )
        raw = f"<html><body>{body}</body></html>".encode()
        mime_type = "text/html"

    request = documentai.ProcessRequest(
        name=processor.name,
        raw_document=documentai.RawDocument(content=raw, mime_type=mime_type),
        process_options=documentai.ProcessOptions(
            layout_config=documentai.ProcessOptions.LayoutConfig(
                chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
                    chunk_size=500,
                    include_ancestor_headings=True,
                )
            )
        ),
    )
    result = docai_client.process_document(request=request)
    return extract_chunks_from_document(result.document)


all_chunks: List[str] = []
for uri in gcs_uris:
    print(f"Processing: {uri}")
    doc_chunks = process_gcs_document(uri)
    print(f"  -> {len(doc_chunks)} chunks extracted")
    all_chunks.extend(doc_chunks)

print(f"\nTotal chunks across all documents: {len(all_chunks)}")
print("\nSample chunk:")
print(all_chunks[0] if all_chunks else "(none)")

## Embed Chunks

Each chunk is embedded with Vertex AI `text-embedding-004`, which produces 768-dimensional vectors. Vectors are L2-normalized so that inner-product similarity equals cosine similarity, which is the metric enVector uses.

In [ ]:
_embed_model = TextEmbeddingModel.from_pretrained("text-embedding-004")


def get_embedding(texts: Union[str, List[str]]) -> np.ndarray:
    """Return L2-normalized float32 embeddings for one or more texts."""
    if isinstance(texts, str):
        texts = [texts]
    embeddings = _embed_model.get_embeddings(texts)
    vecs = np.array([e.values for e in embeddings], dtype=np.float32)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs / norms

In [ ]:
# Embed in batches to stay within the API's per-request limit
BATCH_SIZE = 5
all_vectors = []

for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i : i + BATCH_SIZE]
    vecs = get_embedding(batch)
    all_vectors.append(vecs)
    print(f"Embedded chunks {i + 1}-{i + len(batch)} / {len(all_chunks)}")

vectors = np.vstack(all_vectors)
dim = vectors.shape[1]

print(f"\nVector dimension : {dim}")
print(f"Number of vectors: {vectors.shape[0]}")

## Create Index and Insert

Create an enVector index for the embedding dimension, then insert the chunk vectors.
This is the point where data crosses from the trusted zone into the untrusted enVector server.
`pyenvector` encrypts every vector before transmission — the server stores and operates on ciphertext only.
Chunk text is stored as metadata for retrieval alongside search results.

In [ ]:
index = ev.create_index(INDEX_NAME, dim=dim)
print(f"Index '{INDEX_NAME}' created with dim={dim}")

In [ ]:
# Boundary crossing: pyenvector encrypts each vector before this call.
# The enVector server receives and stores only ciphertext from this point on.
index.insert(vectors, metadata=all_chunks)
print(f"Inserted {len(all_chunks)} encrypted vectors into '{INDEX_NAME}'")

## Encrypted Search

The user query is embedded with the same model and sent to enVector. The server scores it against the encrypted document index and returns encrypted scores, which the client decrypts within the customer's trusted zone to identify the top-k results. The document vectors are stored and searched as ciphertext — the server never decrypts the corpus.

In [ ]:
query_text = "What should I do if search latency is too high?"

query_vector = get_embedding(query_text)[0]  # shape (768,)
print(f"Query vector shape: {query_vector.shape}")

In [ ]:
results = index.search(query_vector, top_k=3, output_fields=["metadata"])[0]

print("Top-3 retrieved chunks:")
for i, res in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(res["metadata"])

## Generate Answer with Gemini

The retrieved chunks are passed to Gemini 2.0 Flash as grounding context. Generation happens entirely within Vertex AI. The enVector server was never involved with plaintext content at any point in the pipeline.

In [ ]:
def generate_answer(docs: List[str], query: str) -> str:
    """Generate an answer grounded in the retrieved document chunks."""
    model = GenerativeModel(GEMINI_MODEL)
    context = "\n\n".join(f"[Document {i+1}]\n{doc}" for i, doc in enumerate(docs))
    prompt = (
        "You are an assistant that answers questions based only on the provided documents.\n\n"
        f"{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    response = model.generate_content(prompt)
    return response.text

In [ ]:
retrieved_docs = [res["metadata"] for res in results]

answer = generate_answer(retrieved_docs, query_text)
print(f"Query: {query_text}")
print(f"\nGenerated Answer:\n{answer}")

## Clean Up

Delete the enVector index and encryption key, then remove the GCS bucket and all its contents. Run this cell when the tutorial is complete to avoid ongoing storage charges.

In [ ]:
ev.drop_index(INDEX_NAME)
print(f"Dropped index '{INDEX_NAME}'")

ev.delete_key(KEY_ID)
print(f"Deleted key '{KEY_ID}'")

In [ ]:
bucket_to_delete = gcs_client.get_bucket(BUCKET_NAME)
bucket_to_delete.delete(force=True)  # force=True deletes all objects first
print(f"Deleted GCS bucket: gs://{BUCKET_NAME}")